In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-8: Where do the weights come from?

## RL for LLMs, Part 2 (RFT alternative methods), session 8 of 14

## What this session teaches

PPO, GRPO, SFT and distillation look like four different methods, but they all do the same thing.

At each token the model's parameters are nudged. The nudge is a number, the weight, times the direction that makes that token more likely.

A positive weight makes the token more likely, a negative weight makes it less likely, and zero leaves it alone.

The methods differ in two ways only.

First, where the weight comes from. It can come from something the model should copy, such as the correct token in a dataset or a teacher's probabilities, or from a reward it should earn, such as a verifier's score.

Second, how fine the weights are: one for a whole answer, one per token, or one for every possible next token.

The deck then shows one consequence. If a model learns from its own answers and every weight is equal, nothing changes on average. Learning from your own output needs a signal from outside.

The point of the deck is that you can look at any training method and ask the same two questions about it: where do its weights come from, and how fine are they?

By the end you can

- say what a weight is, and that every method is "weights times that direction"
- say where a method's weights come from and how fine they are
- say why training on your own answers with equal weights goes nowhere

**Where this session fits: the topics of the series, in order**

<img src="images/rllm8v3/series_strip_7.png" width=88% alt="Strip of the topics of the series with topic 7, where the weights come from, highlighted.">

- this session starts topic 7: where the weights come from


## Setting and words

**Why this slide?** The first sections use these words; each is said by what it does.

| Word | What it is |
|:--|:--|
| **completion** | the whole text the model writes for one prompt (the opening's "answer") |
| **candidate token** | any token the model could write at a position (the opening's "every possible next token") |
| **prefix** | the prompt plus the completion so far (RLLM-7) |
| **direction** | the change of parameters that makes a token likelier: the gradient of its log probability (Recall from Intro B1) |
| **teacher** | a second model, frozen (not trained): it reads the same prefix and gives a probability for every candidate token |
| **reference** | what an update copies: a dataset's token or a teacher's probabilities; not the frozen copy the KL penalty holds the model near (RLLM-5) |
| **verifier**, **reward model** | a program that checks an answer (RLLM-4); a network trained to score answers (RLLM-2) |
| **baseline** | a level subtracted from a reward, so a below-average completion gets a negative weight (RLLM-4, RLLM-5) |

## How we will teach it: the map

**Why this slide?** Five steps, in order; two of them hold cases to keep apart. Each step starts with a picture saying where we are.

<img src="images/rllm8v3/rllm8v3_map.png" width=90% alt="Five boxes in a chain, each with its question: One form, Where the weights come from, How fine, Equal weights, Placing a method.">

- **One form:** what does every update do?
- **Where the weights come from:** two cases, a **reference** to copy or a **reward** to earn
- **How fine:** three cases, **per completion**, **per sampled token** (a token the model wrote) or **per candidate token**
- **Equal weights:** what if nothing outside sets the weights?
- **Placing a method:** the two questions as a grid

## One form: weight times direction, at every token

*You are here: One form.*

<img src="images/rllm8v3/rllm8v3_bar_1.png" width=55% alt="Section bar with One form highlighted.">

**Why this slide?** Without one form, four methods look like four unrelated losses.

<img src="images/rllm8v3/rllm8v3_one_form.png" width=62% alt="Six candidate tokens after the prefix The cat sat on the, each carrying a weight (plus one, minus one, the rest zero) and their probabilities before and after an update.">

- the opening's nudge is the **update**: for each candidate token, its **weight** times its **direction**
- a positive weight makes the token likelier (a **push** up), a negative weight less likely, zero leaves it alone
- a weight is a multiplier, not a parameter

## One form: the same sum in symbols

**Why this slide?** Symbols let later sessions point at one term.

$$\sum_{t}\sum_{v} \; w_t(v)\;\nabla_\theta \log \pi_\theta(v \mid y_{<t})$$

- $\pi_\theta$ is the student and $\theta$ the parameters the update changes
  - the sum is the change added to $\theta$
- $t$ runs over the positions of a batch's completions, $y_{<t}$ is the prefix, $v$ a candidate token
- $w_t(v)$ is the weight of token $v$ at position $t$
- $\nabla_\theta\log\pi_\theta(v\mid y_{<t})$ is the **direction** of token $v$
- a **common form**, not a common objective: each method has its own loss

## Where the weights come from: a reference to copy, or a reward to earn

*You are here: Where the weights come from.*

<img src="images/rllm8v3/rllm8v3_bar_2.png" width=55% alt="Section bar with Where the weights come from highlighted.">

**Why this slide?** The form leaves one thing open, the weight. This is what sets it.

<img src="images/rllm8v3/rllm8v3_source.png" width=50% alt="Two sources of weights: a reference (a dataset's target token, a frozen teacher's probabilities) and a reward (a verifier or a reward model, minus a baseline), both feeding the weights and then the same sum.">

- a **reference** (blue): the dataset's token, or a **teacher**, a frozen model giving a probability for every candidate token
- a **reward** (orange): a verifier's check or a reward model's score, then minus a baseline
- the same two colours return in the grid at the end

## Where the weights come from: copying a dataset's token or a teacher's probabilities

**Why this slide?** "Copy" covers both a dataset and a teacher, and it shows why the sum runs over every candidate.

<img src="images/rllm8v3/rllm8v3_copy.png" width=62% alt="Weights over eight candidate tokens at one position: SFT puts weight 1 on the dataset's token; a teacher gives a probability to every candidate.">

- **SFT** (supervised fine-tuning, trains on a dataset): weight 1 on the dataset's token, 0 on every other candidate
  - the target is the one token with no loss: a one-hot, as in cross-entropy
- **Distillation** (RLLM-1): the teacher reads the same prefix, and its probabilities are the weights
  - including on tokens the student never wrote

## Where the weights come from: earning a reward

**Why this slide?** A reward grades a whole completion; this is how it becomes the same weight for every token.

<img src="images/rllm8v3/rllm8v3_earn.png" width=62% alt="Four completions of one prompt with rewards 1, 0, 0, 1; minus the baseline 0.5 gives weights plus 0.5, minus 0.5, minus 0.5, plus 0.5 on every token.">

- four completions of one prompt earn rewards 1, 0, 0, 1
  - the baseline is the mean, 0.5
- reward minus baseline is the weight: +0.5 or -0.5 on every token of that completion
- this is how PPO and GRPO get weights
  - PPO refines them per token with a critic (RLLM-5)

## Where the weights come from: what you have decides

**Why this slide?** The practical question: which source can you use?

| You have | The weights come from | A limit |
|:--|:--|:--|
| demonstrations (a dataset of good answers, as in SFT) | a **reference**: weight 1 on the dataset's token | our interpretation: a dataset's token cannot mark a sampled completion as bad |
| a teacher model | a **reference**: the teacher's probabilities | a teacher judging the student's own samples can mark them (RLLM-11) |
| a verifier or a reward model | a **reward**, minus a baseline | needs the model to succeed sometimes: a sparse reward is zero for most completions (Recall from RLLM-1) |

## How fine: per completion, per sampled token, per candidate token

*You are here: How fine.*

<img src="images/rllm8v3/rllm8v3_bar_3.png" width=55% alt="Section bar with How fine highlighted.">

**Why this slide?** The second way methods differ: how many numbers the weights carry.

<img src="images/rllm8v3/granularity_strip.png" width=55% alt="Three rows over the tokens of one completion: one weight per completion, one weight per sampled token, a distribution of weights per candidate token.">

- **per completion**: one weight for the whole text, an episode reward
  - **credit assignment** is hard (Recall from Intro A0: deciding which action earned a later reward): which token earned it?
- **per sampled token**: a weight for each token the model wrote
- **per candidate token**: a weight for every token it could have written
  - SFT is the one-hot case

## How fine: a separate question from where the weights come from

**Why this slide?** The same level of fineness can come from different places.

| How fine | Example method | Where the weights come from |
|:--|:--|:--|
| per completion | GRPO (RLLM-6) | the completion's reward minus the group mean: the same weight on every token |
| per sampled token | PPO (RLLM-5) | a critic's advantage estimate; or a teacher's per-token signal on the student's samples (RLLM-11) |
| per candidate token | distillation; SFT is one-hot | a reference: the teacher's probabilities, or the dataset's token |

- one weight per completion is the coarse end: GRPO pays for it with credit assignment, as RLLM-5 noted

## Equal weights: predict what happens

*You are here: Equal weights.*

<img src="images/rllm8v3/rllm8v3_bar_4.png" width=55% alt="Section bar with Equal weights highlighted.">

**Why this slide?** It tests what the weights are for. Try the simplest case: the model trains on its own samples with every weight equal to 1 (a thought experiment, not a method).

**Predict before you advance the slide.** The completions are drawn at temperature 1 (Recall from Intro C1: the model's own probabilities). Averaged over many batches, after one update the model is:

- **better**: it trained on its own reasonable completions
- **worse**: it trained on its own mistakes as well as its successes
- **unchanged, on average**

## Equal weights: no push on average

**Why this slide?** The exact answer. RLLM-7 kept this for today.

<img src="images/rllm8v3/zero_mean_gradient.png" width=58% alt="Same toy policy twice: with equal weights the average push on each logit is zero; with weights equal to a noisy reward minus a baseline it points toward the better action.">

- the model's own samples come in proportion to its probabilities
  - so each token is raised in proportion to its probability, and the probabilities stay where they are
- a reward gives a direction: logit 2 is best, pushed up in **82%** of batches with a reward, **45%** with equal weights
- toy: 4 actions stand for the candidate tokens at one position
  - the push on action $a$ is $e_a-\pi$, where $e_a$ has a 1 at $a$ and 0 elsewhere, and $\pi$ is the vector of probabilities
- the average over the sampled $a$ is $\sum_a\pi_a(e_a-\pi)=0$ (the zero-mean lemma of Intro B1)

## Equal weights: so the weights must come from outside

**Why this slide?** It names what breaks the zero, so the zero is not over-read.

> **Keep this: equal weights on your own completions give no push on average. It returns in RLLM-9 as the reason for a filter (keep only the completions a reward approves).**

- a dataset, a teacher, a verifier or a reward model supplies the weights
- in practice the zero is not exact:
  - a finite batch is noisy (the dots in the picture)
  - Adam (the optimizer that rescales each step) can sharpen the model
  - temperature below 1 or top-p sampling sharpens it
  - GRPO's original Averaging rule divides a completion's weights by its length (RLLM-6), so they are no longer equal
    - our interpretation: that breaks the zero
- "unchanged" means no systematic direction, not no movement at all

<!-- seed-plant: zero-push-equal-weights -->

## Placing a method: the two questions as a grid

*You are here: Placing a method.*

<img src="images/rllm8v3/rllm8v3_bar_5.png" width=50% alt="Section bar with Placing a method highlighted.">

**Why this slide?** The questions become a tool: ask them of any method.

<img src="images/rllm8v3/rllm8v3_grid.png" width=50% alt="Grid: rows others' samples and own samples, columns a reference or a reward sets the weights; SFT and distillation, PPO and GRPO placed, two cells empty.">

- **rows**: whose samples (RLLM-7)
- **columns**: where the weights come from, a reference or a reward
- how fine the weights are is written inside each box
- RLLM-7's other choice, the prefix, sits inside every term of the sum ($y_{<t}$)

## Placing a method: three boxes later sessions fill

**Why this slide?** It tells you where the next three sessions go.

<img src="images/rllm8v3/rllm8v3_slots.png" width=58% alt="The same grid with the empty cells named for later sessions: RLLM-9 own samples both columns, RLLM-10 others' samples with a reward, RLLM-11 own samples with a reference from a teacher.">

- **RLLM-9** (own samples, both columns): a hybrid, a reward picks the completions and those kept are the reference the model trains on
- **RLLM-10** (others' samples, a reward): a preference pair, a better and a worse completion of one prompt (RLLM-2)
- **RLLM-11** (own samples, a reference): a teacher weights every candidate at the student's own prefix, not SFT on the student's own tokens

## Think about

1. You have demonstrations and a verifier for one task. For each, do its weights come from a reference or a reward?
   - what must be true of the model for the verifier's weights to be usable (RLLM-1)?
2. Replace the reward function with a reward *model*. Does where the weights come from change? Does how fine they are change? Argue each.
3. In the RLLM-11 box, *where* would the reference come from? What would you need in hand?
4. All the completions in a batch earn the same reward. After subtracting the average, what are the weights? What happens to the update?

## What we established, and what it sets up

- **Established**
  - one form: a weight times a direction, at every token
  - methods differ in where the weights come from: a reference to copy, a reward to earn
  - and in how fine they are: per completion, per sampled token, per candidate token
  - own completions with equal weights push nowhere on average
  - four methods placed on the grid
- **Limit:** a placement says where the weights come from, not whether it works
- **Sets up the three empty boxes**

| Session | Where it meets this |
|---|---|
| **RLLM-9** sample, filter, imitate | own samples, both columns: a reward picks the data |
| **RLLM-10** DPO | others' samples, a reward, from preference pairs |
| **RLLM-11** on-policy distillation | own samples, a reference from a teacher |

## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Shao et al. (2024), *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*](https://arxiv.org/abs/2402.03300): abstract, sections 4.1 and 5.2.1
  - [Nathan Lambert, *Reinforcement Learning (Motivation & Math)*, lecture slides, rlhfbook.com course](https://rlhfbook.com/teach/course/lec3-chap6-p1/): the policy-gradient derivation, REINFORCE, PPO and GRPO in one slide deck
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved

## Aside, justification: why the average push is zero

*For your own review; not worked through in session.*

- The average push is $\mathbb{E}_{y\sim\pi_\theta}\big[\nabla_\theta\log\pi_\theta(y)\big]=\sum_y \pi_\theta(y)\,\nabla_\theta\log\pi_\theta(y)$
- **Log-derivative trick** (Intro B1, RLLM-5): $\nabla\log\pi=\nabla\pi/\pi$, so the $\pi$'s cancel: $\sum_y\nabla_\theta\pi_\theta(y)=\nabla_\theta\sum_y\pi_\theta(y)=\nabla_\theta 1=0$
- **Toy check** (softmax over 4 actions, parameters = logits $z$): $\nabla_z\log\pi(a)=e_a-\pi$
  - averaging: $\sum_a\pi(a)(e_a-\pi)=\pi-\pi\cdot 1=0$
- **With non-constant weights** $w(a)$, $\sum_a\pi(a)\,w(a)\,(e_a-\pi)\neq 0$ in general: that is the **policy gradient**
- A **constant** weight factors out and gives zero again: subtracting a baseline adds no bias
- All of this holds per token as well as per completion